In [ ]:
!pip install -q kaggle

In [ ]:
from google.colab import files
files.upload()  # Upload kaggle.json


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia


In [ ]:
!unzip chest-xray-pneumonia.zip -d chest_xray_pneumonia


In [ ]:
import os, itertools, random
import numpy as np
import matplotlib.pyplot as plt
import cv2

from sklearn.utils import class_weight
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score, classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.preprocessing.image import ImageDataGenerator


# ------------------------------------------------------------
# 1) Auto-detect dataset root: /content/chest_xray_pneumonia
#    Expecting: train/val/test with NORMAL & PNEUMONIA
# ------------------------------------------------------------
SEARCH_ROOT = "/content/chest_xray_pneumonia"   # change if dataset is elsewhere

def looks_like_dataset(path):
    try:
        for split in ("train", "val", "test"):
            split_path = os.path.join(path, split)
            if not os.path.isdir(split_path):
                return False
            if split == "train":
                if (not os.path.isdir(os.path.join(split_path, "NORMAL")) or
                    not os.path.isdir(os.path.join(split_path, "PNEUMONIA"))):
                    return False
        return True
    except Exception:
        return False

def find_dataset_root(search_root):
    if os.path.isdir(search_root) and looks_like_dataset(search_root):
        return search_root
    if os.path.isdir(search_root):
        # one level down
        for entry in os.listdir(search_root):
            p = os.path.join(search_root, entry)
            if os.path.isdir(p) and looks_like_dataset(p):
                return p
        # walk more levels
        for root, dirs, files in os.walk(search_root):
            if looks_like_dataset(root):
                return root
    return None

DATA_ROOT = find_dataset_root(SEARCH_ROOT)
if DATA_ROOT is None:
    print("Could not find dataset under:", SEARCH_ROOT)
    print("Directory listing for SEARCH_ROOT:",
          os.listdir(SEARCH_ROOT) if os.path.isdir(SEARCH_ROOT) else "Not a directory")
    raise FileNotFoundError(
        "Chest X-ray dataset (train/val/test) not found. "
        "Make sure it is unzipped under /content/chest_xray_pneumonia."
    )

print("Detected dataset root:", DATA_ROOT)

TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR   = os.path.join(DATA_ROOT, "val")
TEST_DIR  = os.path.join(DATA_ROOT, "test")
print("Train dir:", TRAIN_DIR)
print("Val dir:", VAL_DIR)
print("Test dir:", TEST_DIR)

# ------------------------------------------------------------
# 2) Hyperparameters
# ------------------------------------------------------------
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 20       # increase for serious training
FT_EPOCHS = 8
SEED = 42

# ------------------------------------------------------------
# 3) Data generators (DenseNet121 preprocessing)
# ------------------------------------------------------------
train_datagen = ImageDataGenerator(
    preprocessing_function=densenet_preprocess,
    rotation_range=12,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.12,
    horizontal_flip=True,
    fill_mode='nearest'
)
test_datagen = ImageDataGenerator(preprocessing_function=densenet_preprocess)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='rgb',
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
    seed=SEED
)
val_gen = test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='rgb',
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)
test_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='rgb',
    batch_size=1,
    class_mode='binary',
    shuffle=False
)

print("Train samples:", train_gen.samples, "Val samples:", val_gen.samples, "Test samples:", test_gen.samples)
print("Class indices:", train_gen.class_indices)  # e.g. {'NORMAL': 0, 'PNEUMONIA': 1}

# ------------------------------------------------------------
# 4) Class weights (to handle imbalance)
# ------------------------------------------------------------
y_train = train_gen.classes
cw = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = {int(np.unique(y_train)[i]): float(cw[i]) for i in range(len(cw))}
print("Class weights:", class_weights)

# ------------------------------------------------------------
# 5) Build CheXNet-style model (DenseNet121)
# ------------------------------------------------------------
base_model = DenseNet121(weights='imagenet', include_top=False,
                         input_shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
predictions = layers.Dense(1, activation='sigmoid', name='pneumonia_output')(x)

model = models.Model(inputs=base_model.input, outputs=predictions)

# freeze base model initially
for layer in base_model.layers:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

# ------------------------------------------------------------
# 6) Callbacks & checkpoint
# ------------------------------------------------------------
best_model_path = "/content/chexnet_auto_detect_best.h5"   # <--- main .h5 file
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
)
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.2, patience=2, verbose=1
)
checkpoint = keras.callbacks.ModelCheckpoint(
    best_model_path, monitor='val_loss', save_best_only=True, verbose=1
)

# ------------------------------------------------------------
# 7) Train classification head
#    (COMMENT OUT THIS BLOCK IF YOU ALREADY HAVE THE .h5 MODEL)
# ------------------------------------------------------------
history = model.fit(
    train_gen,
    steps_per_epoch=max(1, train_gen.samples // BATCH_SIZE),
    validation_data=val_gen,
    validation_steps=max(1, val_gen.samples // BATCH_SIZE),
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=2
)
print("Head training finished. Epochs run:", len(history.history['loss']))

# ------------------------------------------------------------
# 8) Fine-tune last N layers of DenseNet
#    (COMMENT OUT THIS BLOCK TOO IF MODEL ALREADY TRAINED)
# ------------------------------------------------------------
N_UNFREEZE = 50
for layer in base_model.layers[-N_UNFREEZE:]:
    layer.trainable = True

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_ft = model.fit(
    train_gen,
    steps_per_epoch=max(1, train_gen.samples // BATCH_SIZE),
    validation_data=val_gen,
    validation_steps=max(1, val_gen.samples // BATCH_SIZE),
    epochs=FT_EPOCHS,
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=2
)
print("Fine-tuning finished. Epochs run:", len(history_ft.history['loss']))

# ------------------------------------------------------------
# 9) Load best model & evaluate on test set
# ------------------------------------------------------------
model.load_weights(best_model_path)
print("Loaded best weights from:", best_model_path)

y_true = test_gen.classes
pred_probs = model.predict(test_gen, steps=test_gen.samples).ravel()
y_pred = (pred_probs >= 0.5).astype(int)

print("Test accuracy:", accuracy_score(y_true, y_pred))
try:
    print("Test AUROC:", roc_auc_score(y_true, pred_probs))
except Exception as e:
    print("AUROC could not be computed:", e)

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=list(test_gen.class_indices.keys())))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5,5))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title("Confusion matrix")
plt.colorbar()
ticks = np.arange(len(test_gen.class_indices))
plt.xticks(ticks, list(test_gen.class_indices.keys()), rotation=45)
plt.yticks(ticks, list(test_gen.class_indices.keys()))
thresh = cm.max() / 2.
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    plt.text(j, i, cm[i, j], horizontalalignment="center",
             color="white" if cm[i, j] > thresh else "black")
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.show()